Document Imports

In [206]:
import pandas as pd
import matplotlib.pyplot as plt

### House Sales Dataset: Initial Structure and Descriptive Overview

* This section imports the dataset, inspects its dimensions and data types, and generates descriptive statistics for the numeric variables.

* Skewness is added to the summary table to help identify asymmetric distributions that may require additional attention during preprocessing.

In [207]:
# Load dataset
df = pd.read_csv("house_sales.csv")

# Inspect dataset dimensions and column data types
display(df.shape)
df.info()

# Summarize numeric variables only, excluding non-analytic identifiers
df_num = df.drop(columns=['id', 'date'])

desc = df_num.describe()
desc.loc['skew'] = df_num.skew()

display(desc.round(2).T)

(21613, 21)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       20479 non-null  float64
 4   bathrooms      20545 non-null  float64
 5   sqft_living    20503 non-null  float64
 6   sqft_lot       20569 non-null  float64
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long  

,count,mean,std,min,25%,50%,75%,max,skew
price,21613.0,540088.14,367127.20,75000.00,321950.00,450000.00,645000.00,7700000.00,4.02
bedrooms,20479.0,3.37,0.93,0.00,3.00,3.00,4.00,33.00,2.02
bathrooms,20545.0,2.11,0.77,0.00,1.50,2.25,2.50,8.00,0.50
sqft_living,20503.0,2081.07,915.04,290.00,1430.00,1920.00,2550.00,12050.00,1.39
sqft_lot,20569.0,15179.82,41486.17,520.00,5040.00,7620.00,10708.00,1651359.00,12.88
floors,21613.0,1.49,0.54,1.00,1.00,1.50,2.00,3.50,0.62
waterfront,21613.0,0.01,0.09,0.00,0.00,0.00,0.00,1.00,11.39
view,21613.0,0.23,0.77,0.00,0.00,0.00,0.00,4.00,3.40
condition,21613.0,3.41,0.65,1.00,3.00,3.00,4.00,5.00,1.03
grade,21613.0,7.66,1.18,1.00,7.00,7.00,8.00,13.00,0.77


### Missing Value Inspection

* This section evaluates the extent and structure of missing values in the dataset.  

* Both the frequency of missing observations and the potential co-occurrence of missingness across variables are examined to inform later imputation decisions.

In [208]:
def missing_summary(df):
    """
    Generate summary of missing values.

    Returns count and percentage of missing observations
    for variables with at least one missing entry.
    """
    na_counts = df.isna().sum()
    na_pct = (df.isna().mean() * 100).round(2)

    summary = pd.DataFrame({
        'Missing Count': na_counts,
        'Missing (%)': na_pct
    })

    return summary[summary['Missing Count'] > 0]


# Display missing value summary
display(missing_summary(df))


# Evaluate co-occurrence of missingness across key variables
na_corr = df[['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot']].isna().corr()
display(na_corr)

,Missing Count,Missing (%)
bedrooms,1134,5.25
bathrooms,1068,4.94
sqft_living,1110,5.14
sqft_lot,1044,4.83


,bedrooms,bathrooms,sqft_living,sqft_lot
bedrooms,1.000000,0.002838,0.019517,0.008926
bathrooms,0.002838,1.000000,0.002079,0.014349
sqft_living,0.019517,0.002079,1.000000,-0.006470
sqft_lot,0.008926,0.014349,-0.006470,1.000000


### Duplicate Record Validation

- Verify whether fully duplicated observations exist in the dataset  

- Identify properties with multiple recorded transactions  

- Distinguish between observation-level duplicates and entity-level recurrence  

- Ensure transactional structure aligns with expected real estate market behavior  

In [209]:
# Check to see if any true duplicates exist by row and ID
true_duplicates = df.duplicated().sum()
id_duplicates = (df['id'].value_counts() > 1).sum()
display(f"There are {true_duplicates} true duplicate rows in the data.")
display(f"There are {id_duplicates} houses that sold multiple times.")

'There are 0 true duplicate rows in the data.'

'There are 176 houses that sold multiple times.'

### Date Variable Validation and Conversion

- Confirm timestamp component contains no meaningful variation

- Validate structural integrity of date formatting (YYYYMMDD) 

- Convert string-based date variable to proper datetime type
 
- Determine temporal span of dataset  

In [210]:
# Inspect whether timestamp component varies
print(df['date'].str[-6:].value_counts())

# Validate date structure (month and day ranges)
print(df['date'].str[4:6].astype(int).describe())  # month
print(df['date'].str[6:8].astype(int).describe())  # day

# Convert to datetime format
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

display(df['date'].head())

# Determine temporal coverage of dataset
min_date = df['date'].min()
max_date = df['date'].max()

print(
    f"Dataset spans from {min_date.date()} to {max_date.date()} "
    f"({max_date.date() - min_date.date()} total duration)."
)

date
000000    21613
Name: count, dtype: int64
count    21613.000000
mean         6.574423
std          3.115308
min          1.000000
25%          4.000000
50%          6.000000
75%          9.000000
max         12.000000
Name: date, dtype: float64
count    21613.000000
mean        15.688197
std          8.635063
min          1.000000
25%          8.000000
50%         16.000000
75%         23.000000
max         31.000000
Name: date, dtype: float64


0   2014-10-13
1   2014-12-09
2   2015-02-25
3   2014-12-09
4   2015-02-18
Name: date, dtype: datetime64[ns]

Dataset spans from 2014-05-02 to 2015-05-27 (390 days, 0:00:00 total duration).


### Deterministic Reconstruction of Living Area

- Validate arithmetic identity: **sqft_living = sqft_above + sqft_basement**

- Assess whether missing values can be reconstructed deterministically  

- Apply rule-based imputation where logical constraints are satisfied  

- Re-evaluate consistency and remaining missingness after reconstruction  

In [211]:
# Define function to validate arithmetic identity and reconstruct sqft_living
def validate_and_reconstruct_living(df):
    """
    Validate identity: sqft_living = sqft_above + sqft_basement.
    Reconstruct missing sqft_living values deterministically.
    Returns before/after diagnostics.
    """

    diff_before = (
        df['sqft_living'] - (df['sqft_above'] + df['sqft_basement'])
    ).describe()

    missing_before = df['sqft_living'].isna().sum()

    # Deterministic reconstruction
    df.loc[df['sqft_living'].isna(), 'sqft_living'] = (
        df['sqft_above'] + df['sqft_basement']
    )

    diff_after = (
        df['sqft_living'] - (df['sqft_above'] + df['sqft_basement'])
    ).describe()

    missing_after = df['sqft_living'].isna().sum()

    return diff_before, diff_after, missing_before, missing_after


# Run reconstruction pipeline
before, after, miss_before, miss_after = validate_and_reconstruct_living(df)

print("Consistency check BEFORE reconstruction")
display(before.to_frame(name='Difference'))

print("Consistency check AFTER reconstruction")
display(after.to_frame(name='Difference'))

print(f"Missing sqft_living before: {miss_before}")
print(f"Missing sqft_living after: {miss_after}")

print("\nRemaining missing values in dataset")
display(missing_summary(df))

Consistency check BEFORE reconstruction


,Difference
count,20503.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


Consistency check AFTER reconstruction


,Difference
count,21613.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


Missing sqft_living before: 1110
Missing sqft_living after: 0

Remaining missing values in dataset


,Missing Count,Missing (%)
bedrooms,1134,5.25
bathrooms,1068,4.94
sqft_lot,1044,4.83


### Missingness Mechanism Diagnostics

- Create binary indicators to represent missing observations  

- Evaluate whether missingness is systematically related to key structural or economic variables  

- Assess whether missing values exhibit patterns consistent with MCAR or weak MAR behavior  

- Inform selection of appropriate imputation strategy

In [212]:
# Create missingness indicators
df['bed_missing'] = df['bedrooms'].isna().astype(int)
df['bath_missing'] = df['bathrooms'].isna().astype(int)
df['lot_missing'] = df['sqft_lot'].isna().astype(int)

# Evaluate correlation between missingness and key predictors
missing_corr = df[
    ['bed_missing','bath_missing','lot_missing',
     'price','sqft_living','grade','floors','lat','long']
].corr().loc[
    ['bed_missing','bath_missing','lot_missing'],
    ['price','sqft_living','grade','floors','lat','long']
]

display(missing_corr.round(3))

,price,sqft_living,grade,floors,lat,long
bed_missing,-0.004,-0.007,-0.001,0.003,-0.001,-0.000
bath_missing,-0.000,0.002,0.010,0.007,-0.002,0.013
lot_missing,-0.008,-0.003,-0.003,0.005,-0.001,0.001


### Logical Constraint Validation: Structural Housing Attributes

- Identify records with implausible structural configurations  

- Detect properties reporting zero or negative values for both bedrooms and bathrooms  

- Evaluate whether such observations represent data entry errors, non-residential land, or edge cases requiring exclusion  

- Inform downstream decisions regarding filtering or recoding

In [213]:
# Identify observations with both bedrooms and bathrooms reported as zero or less
invalid_structures = df[(df['bedrooms'] <= 0) & (df['bathrooms'] <= 0)]

display(invalid_structures)

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,bed_missing,bath_missing,lot_missing
875,6306400140,2014-06-12,1095000.0,0.0,0.0,3064.0,4764.0,3.5,0,2,...,1990,0,98102,47.6362,-122.322,2360,4000,0,0,0
6994,2954400190,2014-06-24,1295650.0,0.0,0.0,4810.0,28008.0,2.0,0,0,...,1990,0,98053,47.6642,-122.069,4740,35061,0,0,0
9773,3374500520,2015-04-29,355000.0,0.0,0.0,2460.0,8049.0,2.0,0,0,...,1990,0,98031,47.4095,-122.168,2520,8050,0,0,0
9854,7849202190,2014-12-23,235000.0,0.0,0.0,1470.0,4800.0,2.0,0,0,...,1996,0,98065,47.5265,-121.828,1060,7200,0,0,0
